# Evaluation Node: Theoretical Orientation
Evaluate `evaluation_theoretical_orientation` outputs against benchmark labels and rerun this node on selected papers.

In [3]:
import os
import sys
import json
import textwrap
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / 'nodes').exists():
        break
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent
if not (ROOT / 'nodes').exists():
    raise RuntimeError(f"Could not resolve project root from cwd: {Path.cwd()}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.dsrp_state import DSRPState

PILOT = ROOT / 'Pilot_Evaluation'
BENCHMARK_FILE = PILOT / 'DATA_sample_10' / 'Data Science Research Process (DSRP) Framework.csv'
RESULTS_FOLDER = PILOT / 'pilot_study_results'
VECTOR_DB_PATH = PILOT / 'pilot_study_10_vectors'
COLLECTION_NAME = 'pilot_study_10'
EMBEDDING_MODEL = 'text-embedding-3-small'
NODE_KEY = 'evaluation_theoretical_orientation'

DIMENSIONS = [
    'explicit_theory',
    'implicit_theory_detected',
    'epistemological_orientation',
    'primary_research_orientation',
]

In [4]:
def normalize_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

def normalize_paper_id(value):
    text = normalize_text(value)
    return text.lower() if text else None

def canonical_yes_no(value):
    text = normalize_text(value)
    if not text:
        return 'No'
    return 'Yes' if text.lower() in {'yes', 'true', '1', 'y'} else 'No'

def canonical_epistemological(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        'theory-driven research': 'Theory-driven research',
        'hybrid (data + theory)': 'Hybrid (data + theory)',
        'data-driven discovery': 'Data-driven discovery',
    }
    return mapping.get(text.lower(), text)

def canonical_primary_orientation(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        'method-oriented research': 'Method-oriented research',
        'knowledge-oriented research': 'Knowledge-oriented research',
        'hybrid (method + knowledge)': 'Hybrid (method + knowledge)',
    }
    return mapping.get(text.lower(), text)

def normalize_dimension(dim, value):
    if dim in {'explicit_theory', 'implicit_theory_detected'}:
        return canonical_yes_no(value)
    if dim == 'epistemological_orientation':
        return canonical_epistemological(value)
    if dim == 'primary_research_orientation':
        return canonical_primary_orientation(value)
    return value

def load_agent_results(results_folder):
    data = {}
    for paper_dir in sorted(results_folder.glob('*/')):
        results_file = paper_dir / 'aggregated' / 'results.json'
        if not results_file.exists():
            continue
        with open(results_file, 'r', encoding='utf-8') as f:
            parsed = json.load(f)
            pid = normalize_paper_id(parsed.get('paper_id', paper_dir.name))
            if pid:
                data[pid] = parsed.get('dsrp_outputs', {})
    return data

In [5]:
if not BENCHMARK_FILE.exists():
    raise FileNotFoundError(f"Ground truth file not found: {BENCHMARK_FILE}")

csv_mtime = pd.to_datetime(BENCHMARK_FILE.stat().st_mtime, unit='s')
print(f"Reading fresh ground truth from: {BENCHMARK_FILE}")
print(f"CSV last modified: {csv_mtime}")

def load_benchmark_eval_from_csv(verbose=True):
    if not BENCHMARK_FILE.exists():
        raise FileNotFoundError(f"Ground truth file not found: {BENCHMARK_FILE}")

    csv_mtime = pd.to_datetime(BENCHMARK_FILE.stat().st_mtime, unit='s')
    fresh_df = pd.read_csv(BENCHMARK_FILE, low_memory=False)
    fresh_df.columns = fresh_df.columns.str.strip()
    required_cols = ['Paper ID', 'Gatekeeper'] + DIMENSIONS
    missing_cols = [c for c in required_cols if c not in fresh_df.columns]
    if missing_cols:
        raise ValueError(f'Missing required benchmark columns: {missing_cols}')

    fresh_eval = fresh_df[required_cols].copy()
    fresh_eval['Paper ID'] = fresh_eval['Paper ID'].apply(normalize_paper_id)
    fresh_eval['Gatekeeper'] = fresh_eval['Gatekeeper'].astype(str).str.strip().str.lower()
    for dim in DIMENSIONS:
        fresh_eval[dim] = fresh_eval[dim].apply(lambda v: normalize_dimension(dim, v))
    fresh_eval = fresh_eval[fresh_eval['Gatekeeper'] == 'include'].copy()
    fresh_eval = fresh_eval.dropna(subset=['Paper ID']).reset_index(drop=True)

    if verbose:
        print(f"Ground truth reloaded from CSV: {BENCHMARK_FILE}")
        print(f"CSV last modified: {csv_mtime}")
        print(f"Included benchmark rows: {len(fresh_eval)}")

    return fresh_eval

benchmark_eval = load_benchmark_eval_from_csv(verbose=True)

agent_data = load_agent_results(RESULTS_FOLDER)
comparison_rows = []
for _, row in benchmark_eval.iterrows():
    pid = row['Paper ID']
    if pid not in agent_data:
        continue
    node_output = agent_data[pid].get(NODE_KEY, {})
    record = {'Paper ID': pid}
    flags = []
    for dim in DIMENSIONS:
        gt = row[dim]
        pred = normalize_dimension(dim, node_output.get(dim))
        match = gt == pred
        flags.append(match)
        record[f'GT_{dim}'] = gt
        record[f'Agent_{dim}'] = pred
        record[f'Match_{dim}'] = 'MATCH' if match else 'MISMATCH'
    record['Match_All'] = 'MATCH' if all(flags) else 'MISMATCH'
    comparison_rows.append(record)

comparison_df = pd.DataFrame(comparison_rows)
print(f'Comparable rows: {len(comparison_df)}')
show_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
print(comparison_df[show_cols].to_string(index=False))

Reading fresh ground truth from: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-08 16:42:55.407081366
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-08 16:42:55.407081366
Included benchmark rows: 7
Comparable rows: 7
 Paper ID Match_explicit_theory Match_implicit_theory_detected Match_epistemological_orientation Match_primary_research_orientation Match_All
 2018 - 8                 MATCH                       MISMATCH                          MISMATCH                           MISMATCH  MISMATCH
2019 - 33                 MATCH                          MATCH                             MATCH                              MATCH     MATCH
 2020 - 8                 MATCH                          

In [6]:
metrics_rows = []
for dim in DIMENSIONS:
    y_true = comparison_df[f'GT_{dim}'].tolist()
    y_pred = comparison_df[f'Agent_{dim}'].tolist()
    metrics_rows.append({
        'dimension': dim,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    })
metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string(index=False))

                   dimension  accuracy  precision_macro  recall_macro  f1_macro
             explicit_theory  0.714286         0.666667      0.833333  0.650000
    implicit_theory_detected  0.285714         0.500000      0.142857  0.222222
 epistemological_orientation  0.428571         0.416667      0.444444  0.300000
primary_research_orientation  0.714286         0.800000      0.750000  0.708333


## Iteration Utility

In [7]:
import importlib
import nodes.evaluation_nodes.evaluation_theoretical_orientation as node_module

def _extract_theoretical_reasoning(node_output):
    if not isinstance(node_output, dict):
        return '(no reasoning found)', ''
    reasoning = (
        node_output.get('validated_reasoning')
        or node_output.get('reasoning_explanation')
        or node_output.get('reasoning')
        or '(no reasoning found)'
    )
    audit = node_output.get('audit_commentary', '')
    return str(reasoning), str(audit or '')

def _extract_theoretical_bibliography(node_output):
    if not isinstance(node_output, dict):
        return []
    bib = node_output.get('validated_bibliography')
    if isinstance(bib, list):
        return bib
    bib = node_output.get('bibliography')
    if isinstance(bib, list):
        return bib
    return []

def _run_theoretical_iteration_base(paper_ids, llm_model='gpt-4o-mini', verbose=True):
    importlib.reload(node_module)
    theoretical_node = node_module.evaluation_theoretical_orientation_node

    benchmark_lookup = benchmark_eval.set_index('Paper ID')
    rows = []
    raw_outputs = {}
    original_dir = Path.cwd().resolve()
    prev_env_model = os.getenv('DSRP_LLM_MODEL')

    os.chdir(ROOT)
    os.environ['DSRP_LLM_MODEL'] = llm_model

    try:
        for paper_id in paper_ids:
            pid = normalize_paper_id(paper_id)
            if not pid:
                continue

            if pid not in benchmark_lookup.index:
                rows.append({'Paper ID': pid, 'Model': llm_model, 'Error': 'Paper ID not found in benchmark include set'})
                continue

            state = DSRPState(
                paper_id=pid,
                dsrp_outputs={},
                collection_name=COLLECTION_NAME,
                persist_directory=str(VECTOR_DB_PATH),
                embedding_model=EMBEDDING_MODEL,
                gatekeeper={},
            )
            try:
                out = theoretical_node(state)
                dsrp_outputs = out.get('dsrp_outputs', state['dsrp_outputs'])
                node_output = dsrp_outputs.get(NODE_KEY, {})
                raw_outputs[pid] = node_output

                record = {'Paper ID': pid, 'Model': llm_model}
                flags = []
                for dim in DIMENSIONS:
                    gt = benchmark_lookup.loc[pid, dim]
                    pred = normalize_dimension(dim, node_output.get(dim))
                    match = gt == pred
                    flags.append(match)
                    record[f'GT_{dim}'] = gt
                    record[f'New_Agent_{dim}'] = pred
                    record[f'Match_{dim}'] = 'MATCH' if match else 'MISMATCH'
                record['Match_All'] = 'MATCH' if all(flags) else 'MISMATCH'
                rows.append(record)

                if verbose:
                    summary = ' | '.join([f"{d}: {record[f'Match_{d}']}" for d in DIMENSIONS])
                    print(f"{pid}: {summary} | Match_All: {record['Match_All']}")
            except Exception as e:
                rows.append({'Paper ID': pid, 'Model': llm_model, 'Error': str(e)})
                if verbose:
                    print(f'{pid}: ERROR -> {e}')
    finally:
        if prev_env_model is None:
            os.environ.pop('DSRP_LLM_MODEL', None)
        else:
            os.environ['DSRP_LLM_MODEL'] = prev_env_model
        os.chdir(original_dir)

    return pd.DataFrame(rows), raw_outputs

def run_theoretical_iteration(paper_ids=None, llm_model='gpt-4o-mini', verbose=True):
    global benchmark_eval

    old_all_included = []
    if isinstance(benchmark_eval, pd.DataFrame) and 'Paper ID' in benchmark_eval.columns:
        old_all_included = benchmark_eval['Paper ID'].tolist()

    benchmark_eval = load_benchmark_eval_from_csv(verbose=verbose)

    if paper_ids is None or paper_ids == old_all_included:
        paper_ids = benchmark_eval['Paper ID'].tolist()

    return _run_theoretical_iteration_base(
        paper_ids=paper_ids,
        llm_model=llm_model,
        verbose=verbose,
    )

def show_theoretical_label_comparison(df):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        print('No results to display.')
        return
    cols = ['Paper ID', 'Model']
    for dim in DIMENSIONS:
        cols.extend([f'GT_{dim}', f'New_Agent_{dim}', f'Match_{dim}'])
    if 'Match_All' in df.columns:
        cols.append('Match_All')
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))

def view_existing_theoretical_reasoning(retest_df, retest_outputs, paper_id, show_bibliography=False):
    pid = normalize_paper_id(paper_id)
    rows = retest_df[retest_df['Paper ID'].apply(normalize_paper_id) == pid]
    if len(rows) == 0:
        raise ValueError(f'Paper ID not found in retest_df: {paper_id}')

    row = rows.iloc[-1].to_dict()
    node_output = retest_outputs.get(pid, {})

    def _fmt(value):
        if isinstance(value, list):
            return ', '.join(str(v) for v in value) if value else '(none)'
        if isinstance(value, dict):
            return json.dumps(value, indent=2)
        if value is None:
            return '(missing)'
        return str(value)

    def _print_wrapped(title, text, indent='  ', width=110):
        print(title)
        txt = str(text or '').strip()
        if not txt:
            print(f"{indent}(none)")
            return
        for line in txt.splitlines() or ['']:
            if not line.strip():
                print('')
                continue
            print(textwrap.fill(line, width=width, initial_indent=indent, subsequent_indent=indent))

    print('\n' + '=' * 100)
    print(f"PAPER: {row.get('Paper ID', '(missing)')} | MODEL: {row.get('Model', '(missing)')}")
    print('=' * 100)
    for dim in DIMENSIONS:
        print(f"{dim}")
        print(f"  GT    : {_fmt(row.get(f'GT_{dim}', '(missing)'))}")
        print(f"  Agent : {_fmt(row.get(f'New_Agent_{dim}', '(missing)'))}")
        print(f"  Match : {row.get(f'Match_{dim}', '(missing)')}")
    print(f"\nMatch_All: {row.get('Match_All', '(missing)')}")

    reasoning, audit = _extract_theoretical_reasoning(node_output)
    _print_wrapped('\nReasoning:', reasoning)
    if audit.strip():
        _print_wrapped('\nAudit Commentary:', audit)

    if show_bibliography:
        bibliography = _extract_theoretical_bibliography(node_output)
        print('\nBibliography:')
        if not bibliography:
            print('  (none)')
        else:
            for i, item in enumerate(bibliography, start=1):
                item_id = item.get('id', i) if isinstance(item, dict) else i
                page = item.get('page', '') if isinstance(item, dict) else ''
                section = item.get('section', '') if isinstance(item, dict) else ''
                quote = item.get('direct_quote', '') if isinstance(item, dict) else ''
                print(f"  [{item_id}] page={page} | section={section}")
                _print_wrapped('      quote:', quote, indent='        ', width=105)

def rerun_single_theoretical_paper(paper_id, llm_model='gpt-4o-mini', verbose=True, show_bibliography=False):
    df, outputs = run_theoretical_iteration([paper_id], llm_model=llm_model, verbose=verbose)
    show_theoretical_label_comparison(df)
    print('\nReadable single-paper output:')
    view_existing_theoretical_reasoning(
        retest_df=df,
        retest_outputs=outputs,
        paper_id=paper_id,
        show_bibliography=show_bibliography,
    )
    return df, outputs

inspect_theoretical_reasoning = view_existing_theoretical_reasoning

Iteraton 1

In [6]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-07 13:29:00.078239918
Included benchmark rows: 7


C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


2018 - 8: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2019 - 33: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2020 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MISMATCH | Match_All: MISMATCH
2021 - 28: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2021 - 56: explicit_theory: MATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2023 - 58: explicit_theory: MISMATCH | implicit_theory_detected: MATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Ma

After proper bibliography and citations

In [10]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-07 13:29:00.078239918
Included benchmark rows: 7
2018 - 8: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2019 - 33: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2020 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MISMATCH | Match_All: MISMATCH
2021 - 28: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2021 - 56: explicit_theory:

Iteration 3: Added Clear Rules for Implicit Explicit Classification | Added Evidence Alignment Instruction in Auditor.

In [21]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-07 13:29:00.078239918
Included benchmark rows: 7
2018 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2019 - 33: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2020 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MISMATCH | Match_All: MISMATCH
2021 - 28: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2021 - 56: explicit_theory: MATCH | imp

re-test

In [22]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-07 13:29:00.078239918
Included benchmark rows: 7
2018 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2019 - 33: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2020 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MISMATCH | Match_All: MISMATCH
2021 - 28: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2021 - 56: explicit_theory: MATCH | imp

Added More rules to correctly label 'Explicit Theory'

In [28]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-08 16:20:24.281978846
Included benchmark rows: 7
2018 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MISMATCH | Match_All: MISMATCH
2019 - 33: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2020 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2021 - 28: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2021 - 56: explicit_theory: MATCH | implicit_theory

re-test

In [39]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-08 16:42:55.407081366
Included benchmark rows: 7
2018 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2019 - 33: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2020 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MISMATCH | Match_All: MISMATCH
2021 - 28: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2021 - 56: explicit_theory: MATCH | implicit_theory

re-test

In [9]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-08 16:42:55.407081366
Included benchmark rows: 7
2018 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2019 - 33: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2020 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MISMATCH | Match_All: MISMATCH
2021 - 28: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2021 - 56: explicit_theory: MATCH | implicit_theory

### Iteration: With Changed LLM (gpt-4.1-nano)

In [20]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4.1-nano', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-09 02:21:24.491600037
Included benchmark rows: 7
2018 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2019 - 33: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2020 - 8: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2021 - 28: explicit_theory: MISMATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2021 - 56: explicit_t

Iteration: With Changed LLM (gpt-5-nano)

In [21]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-5-nano', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-09 02:21:24.491600037
Included benchmark rows: 7
2018 - 8: explicit_theory: MATCH | implicit_theory_detected: MISMATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
2019 - 33: ERROR -> LLM returned invalid JSON: Expecting ',' delimiter: line 7 column 449 (char 660)
Raw:
{
  "explicit_theory": "No",
  "implicit_theory_detected": "Yes",
  "epistemological_orientation": "Hybrid (data + theory)",
  "primary_research_orientation": "Knowledge-oriented research",
  "confidence": 0.65,
  "reasoning_explanation": "Step 1 (explicit theory): The study discusses the role of submission devices and references theoretical considerations about mobile ecosystems, but there is no evidence that a named theor

KeyboardInterrupt: 

### Iteration: Back to LLM Model 4o Mini Because there is huge deviation with other similar small models. 

In [22]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_theoretical_iteration(all_included_papers, llm_model='gpt-4o-mini', verbose=True)

print('\nDetailed label comparison (GT vs Agent):')
show_theoretical_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-09 02:21:24.491600037
Included benchmark rows: 7
2018 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2019 - 33: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2020 - 8: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2021 - 28: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MATCH | primary_research_orientation: MATCH | Match_All: MATCH
2021 - 56: explicit_theory: MATCH | implicit_theory_detec

In [23]:
# Helper usage examples

# A) View reasoning from existing retest results (NO rerun)
view_existing_theoretical_reasoning(
    retest_df=retest_df,
    retest_outputs=retest_outputs,
    paper_id='2024 - 99',
    show_bibliography=True,
 )




PAPER: 2024 - 99 | MODEL: gpt-4o-mini
explicit_theory
  GT    : Yes
  Agent : No
  Match : MISMATCH
implicit_theory_detected
  GT    : Yes
  Agent : No
  Match : MISMATCH
epistemological_orientation
  GT    : Theory-driven research
  Agent : Hybrid (data + theory)
  Match : MISMATCH
primary_research_orientation
  GT    : Hybrid (method + knowledge)
  Agent : Knowledge-oriented research
  Match : MISMATCH

Match_All: MISMATCH

Reasoning:
  The study demonstrates explicit theory by integrating Modern Portfolio Theory (MPT) and normative theory
  into its framework for climate adaptation planning, which directly structures the model and guides variable
  selection. The explicit use of these theories is evident in the research objectives and the development of a
  decision-making model. Additionally, the results are interpreted through the lens of sustainability and
  cultural heritage, indicating implicit theoretical reasoning as real-world mechanisms are employed to
  explain findings. 

In [24]:
# B) Re-run on a single paper only (optional)
single_retest_df, single_retest_outputs = rerun_single_theoretical_paper(
     paper_id='2024 - 99',
     llm_model='gpt-4o-mini',
     verbose=True,
     show_bibliography=True,
 )

Ground truth reloaded from CSV: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-09 02:21:24.491600037
Included benchmark rows: 7
2024 - 99: explicit_theory: MATCH | implicit_theory_detected: MATCH | epistemological_orientation: MISMATCH | primary_research_orientation: MATCH | Match_All: MISMATCH
 Paper ID       Model GT_explicit_theory New_Agent_explicit_theory Match_explicit_theory GT_implicit_theory_detected New_Agent_implicit_theory_detected Match_implicit_theory_detected GT_epistemological_orientation New_Agent_epistemological_orientation Match_epistemological_orientation GT_primary_research_orientation New_Agent_primary_research_orientation Match_primary_research_orientation Match_All
2024 - 99 gpt-4o-mini                Yes                       Yes                 MATCH                         Yes                                Yes                 